# Visual search with a vision-language model

Qwen3-VL-2B-Thinking looks at a search display, thinks, then answers yes or no. Its reaction time
is the number of tokens it thinks for. The displays are drawn by `stimuli.py` in this folder; the
question is asked by `observer.py`.

- **Tasks:** a red bar among green bars, and a 2 among 5s, as in Wolfe, Palmer & Horowitz (2010)
- **Set sizes:** 3, 6, 12 and 18 items; 100 layouts each, every layout once with the target and
  once without it: 1600 displays

Results go to `../data/model_results.csv`, one row per display, with the full thinking text.
`analysis.ipynb` makes the numbers and figures from that file.

Needs a GPU. In Colab: Runtime > Change runtime type > **T4 GPU** (the setup cell stops if there is none).

## Setup

In [1]:
import os
import sys

inColab = "google.colab" in sys.modules

if inColab:
  from google.colab import drive
  drive.mount("/content/drive")
  os.chdir("/content/drive/MyDrive/Projects/vlm-psychophysics/chapters/visual_search/code")
  os.system("pip install -q -U transformers")

sys.path.insert(0, os.getcwd())
print("running in Colab:", inColab)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
running in Colab: True


In [2]:
import time
import torch
import numpy  as np
import pandas as pd
from transformers.utils import logging

import observer
import stimuli

logging.disable_progress_bar()
logging.set_verbosity_error()

assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"
obs = observer.Observer(device="cuda")
print("model:", observer.MODEL, "| GPU:", torch.cuda.get_device_name())
print("set sizes:", stimuli.SET_SIZES, "| layouts per task and set size:", stimuli.PAIRS)

model: Qwen/Qwen3-VL-2B-Thinking | GPU: Tesla T4
set sizes: [3, 6, 12, 18] | layouts per task and set size: 100


## The question

The chat as the model gets it. The image is one block of `<|image_pad|>` tokens; the model's reply
starts after `<think>`. Its thinking ends with `</think>`, then comes its answer.

In [3]:
for task in stimuli.TASKS:
  print(obs.proc.apply_chat_template(obs.chat(task), tokenize=False, add_generation_prompt=True))

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Is there a red bar in the image? Answer yes or no.<|im_end|>
<|im_start|>assistant
<think>

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Is there a 2 in the image? Answer yes or no.<|im_end|>
<|im_start|>assistant
<think>



## The scoring

- **RT:** the number of tokens before `</think>`.
- **Answer:** the first "yes" or "no" after `</think>`. A display is **correct** if the answer is
  "yes" with the target there, "no" without it.
- A reply still thinking after 4096 tokens is stopped and counted as no answer.

## The run

One batch = displays of one task and set size, each layout with and without the target: 16 displays
for the red bar, 4 for the 2 among 5s, whose long replies would fill the GPU's memory at 16.
The replies are sampled, with the random seed set from the first display of each batch, so a rerun
gives the same replies. If the run stops, run the cell again: displays already in the file are skipped.

In [4]:
resultsFile = "../data/model_results.csv"
BATCH       = {"feature": 16, "2v5": 4}

results = pd.read_csv(resultsFile).to_dict("records") if os.path.exists(resultsFile) else []
done    = set(r["index"] for r in results)
todo    = [t for t in stimuli.make_trials() if t["index"] not in done]
print("%d displays done, %d to go" % (len(done), len(todo)))

startTime = time.time()
groups = {}
for t in todo:
  groups.setdefault((t["task"], t["set_size"]), []).append(t)
for (task, n), group in groups.items():
  for b in range(0, len(group), BATCH[task]):
    batch   = group[b:b + BATCH[task]]
    answers = obs.ask(task, [stimuli.draw_trial(t) for t in batch], seed=batch[0]["index"])
    for trial, a in zip(batch, answers):
      said = {"yes": 1, "no": 0}.get(a["answer"])
      results.append({**trial, **a, "model": observer.MODEL,
                      "correct": int(said == trial["present"]) if said is not None else np.nan})
    pd.DataFrame(results).to_csv(resultsFile, index=False)
  print("%-8s %2d items done | %d displays, %.0f min" % (task, n, len(results), (time.time() - startTime) / 60),
        flush=True)

print("done: %d displays in %.0f min" % (len(results), (time.time() - startTime) / 60))

880 displays done, 720 to go
2v5       3 items done | 1000 displays, 5 min
2v5       6 items done | 1200 displays, 13 min
2v5      12 items done | 1400 displays, 27 min
2v5      18 items done | 1600 displays, 40 min
done: 1600 displays in 40 min


In [5]:
df = pd.DataFrame(results)
print(df.groupby(["task", "present", "set_size"]).agg(
    correct=("correct", "mean"), rt_median=("rt_tokens", "median"), rt_mean=("rt_tokens", "mean"),
    unfinished=("finished", lambda f: int((f == 0).sum())), displays=("index", "count")).round(2).to_string())

                          correct  rt_median  rt_mean  unfinished  displays
task    present set_size                                                   
2v5     0       3            1.00       46.5    57.46           0       100
                6            1.00       53.0    56.91           0       100
                12           1.00       46.0    45.61           0       100
                18           1.00       47.0    49.22           0       100
        1       3            1.00       50.0    51.61           0       100
                6            0.95       56.5    68.38           0       100
                12           0.88       69.0   118.14           0       100
                18           0.87       76.5   117.54           0       100
feature 0       3            1.00       36.0    39.24           0       100
                6            1.00       43.0    40.00           0       100
                12           1.00       43.0    39.63           0       100
            